# 1D 切割下料 TEST0005 —— 对偶/割推导总览（Gilmore-Gomory 结构）

问题：$\min\{\sum_p x_p:\ \sum_p a_{ip}x_p\ge d_i,\ x_p\in\mathbb{Z}_+\}$，列=切割模式
（$\sum l_i a_i\le 10000$），最优 28（长度下界 28 = 28 辊解）。

## 01 直接建模 —— 无对偶
证明链：总长度下界 28 + 池 MIP 28 辊解（覆盖校验）；CP-SAT item→bin 直接模型 K=28 难构造（UNKNOWN 560s）。

## 02 列生成 —— LP 对偶
RMP（min Σx）：对偶 $\max\sum_i d_i\pi_i$ s.t. $\sum_i a_{ip}\pi_i\le 1\ \forall$模式, $\pi_i\ge0$。
reduced cost $rc_p=1-\sum_i a_{ip}\pi_i$；定价 = 0/1 背包 $\max\sum\pi_i a_i$（numpy DP）；
无负 rc ⇒ LP 最优（27.9942，对偶=原值强对偶）。

## 03 Benders —— 子问题对偶 → 割
SP(y) 对偶：$\max\sum_i d_i\pi_i+\sum_p\sigma_p(M y_p)$，s.t. $\sum_i a_{ip}\pi_i+\sigma_p\le 1$；
弱对偶 ⇒ 割 $\theta+\sum\lambda_p y_p\ge\sum d_i\pi_i$（λ=−σ）。

## 04 拉格朗日 —— 乘子对偶 + 次梯度
$L(\lambda)=\sum_i d_i\lambda_i+M\cdot\min(0,\ 1-v(\lambda))$，$v(\lambda)$=背包最优值；
子问题=背包（无需求解器）；次梯度 $g_i=d_i-\sum_p a_{ip}x_p$；对偶 → LP 27.9942。
（注意 x 需有界 M=28，否则 rc<0 时对偶函数 −∞。）

## 05 LBBD —— 逻辑割（1D 退化）
主问题 item→bin；子问题=容量检查（线性）；超载箱回传惰性容量割
$\sum_{i\in S}x_{ib}\le|S|-1$（最小超载核心）——1D 下 LBBD 退化为惰性约束主问题（CONVENTIONS §4.5）。

## 07 Branch-and-Price —— 辊数分支
根 LP 27.9942 分数 → 分支 Σx≤27（长度下界不可行）vs Σx≥28（节点整数恢复 28）；
GG pair 分支框架（forbid=双 DP 排除端点 / merge=超件合并）已实现于 scripts/csp_bnp.py。


In [1]:
# 数值验证：RMP 对偶 + 强对偶 + 定价值 = 1（对偶可行）
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
import sys, platform
import ortools
sys.path.insert(0, "/mnt/d/exactTest/column-generation-solvers/csp_1d_waescher/scripts")
import csp_core as cc
from ortools.math_opt.python import mathopt
print("python", platform.python_version(), "| ortools", ortools.__version__)

lp, patterns, sel, iters = cc.cg_min_rolls()
# 最终对偶：重解 RMP 取 π
mm = mathopt.Model()
x = [mm.add_variable(lb=0.0, ub=float("inf"), is_integer=False, name=f"x{k}") for k in sel]
covers = []
for i in range(cc.M):
    covers.append(mm.add_linear_constraint(
        mathopt.fast_sum([x[k]*patterns[k][i] for k in range(len(sel))]) >= cc.DEM[i], name=f"c{i}"))
mm.minimize(mathopt.fast_sum(x))
res = mathopt.solve(mm, mathopt.SolverType.GLOP)
dv = res.dual_values()
pi = [max(0.0, dv[covers[i]]) for i in range(cc.M)]
dual_obj = sum(pi[i]*cc.DEM[i] for i in range(cc.M))
print(f"LP 目标 = {round(lp,6)} | 对偶目标 Σdπ = {round(dual_obj,6)} | 强对偶: {abs(lp-dual_obj)<1e-6}")
v, pat = cc.knap_rebuild(pi)
print(f"定价背包最优值 v(π) = {round(v,6)} | 对偶约束 Σaπ<=1 满足: {v <= 1 + 1e-9}")
rc = 1.0 - v
print(f"最负 rc = {round(rc,8)}（≈0 ⇒ 无改进列 ⇒ LP 最优）")


python 3.10.20 | ortools 9.15.6755


LP 目标 = 27.994174 | 对偶目标 Σdπ = 27.994174 | 强对偶: True
定价背包最优值 v(π) = 1.0 | 对偶约束 Σaπ<=1 满足: True
最负 rc = -0.0（≈0 ⇒ 无改进列 ⇒ LP 最优）
